# Analysis — ECB Rate Shock & SME Failures

## Objective
Estimate, by difference-in-differences, whether PME failures accelerated more in bank-credit-dependent sectors after the July 2022 ECB rate hike, relative to the 2015-2019 baseline. The outcome is the log of the 12-month rolling failure count; standard errors are clustered by sector.

In [ ]:
from pathlib import Path
import sys
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import statsmodels.formula.api as smf
import warnings
from statsmodels.tools.sm_exceptions import ValueWarning
warnings.simplefilter('ignore', ValueWarning)
from IPython.display import display

sys.path.insert(0, str(Path().resolve().parent))
from src import cleaning

Path('../outputs').mkdir(exist_ok=True)
panel = pd.read_csv('../data/processed/sector_panel.csv', parse_dates=['date'])
treatment = pd.read_csv('../data/processed/treatment.csv', index_col=0)
sample = panel[panel['in_sample']].copy()
sample['log_failures'] = np.log(sample['failures'])
print('sample rows:', len(sample), '| sectors:', sample['sector'].nunique())
print('treated sectors:', sample.loc[sample['treated'] == 1, 'sector'].nunique(),
      '| control sectors:', sample.loc[sample['treated'] == 0, 'sector'].nunique())

## 1. Main difference-in-differences
Sector fixed effects absorb time-invariant sector levels; `post` captures the common shift after July 2022; `did` (= treated x post) is the causal estimate. With only 9 sectors the cluster-robust standard errors rest on few clusters and should be read with caution.

In [ ]:
model = smf.ols('log_failures ~ C(sector) + post + did', data=sample).fit(
    cov_type='cluster', cov_kwds={'groups': sample['sector']})
print(model.summary().tables[1])

coef = model.params['did']
print(f'\nDiD coefficient (log points): {coef:+.3f}')
print(f'Implied relative effect on failures: {np.expm1(coef) * 100:+.1f}%')

**Results from the main DiD:** the `did` coefficient is the additional change in (log) failures for bank-credit-dependent sectors after the rate hike, on top of the common post-2022 movement. Read as a relative effect via exp(coef) - 1. The few-clusters caveat applies to its significance.

## 2. Robustness — alternative treatment threshold
Instead of the median split, take the top-3 most bank-dependent sectors as treatment and the bottom-3 as control, dropping the middle three. A real effect should survive a sharper contrast.

In [ ]:
ranked = treatment['bank_dependence'].sort_values(ascending=False)
top3, bottom3 = list(ranked.index[:3]), list(ranked.index[-3:])
sub = sample[sample['sector'].isin(top3 + bottom3)].copy()
sub['treated_t'] = sub['sector'].isin(top3).astype(int)
sub['did_t'] = sub['treated_t'] * sub['post']
m2 = smf.ols('log_failures ~ C(sector) + post + did_t', data=sub).fit(
    cov_type='cluster', cov_kwds={'groups': sub['sector']})
print('treated (top3):', top3, '| control (bottom3):', bottom3)
print(m2.summary().tables[1])
print(f"\nDiD (tertile split): {m2.params['did_t']:+.3f} log pts "
      f"({np.expm1(m2.params['did_t']) * 100:+.1f}%)")

## 3. Placebo — fake hike inside the baseline
Using only the clean 2015-2019 baseline, assign a fake treatment date of January 2018. A genuine rate effect should leave this placebo near zero and insignificant.

In [ ]:
base = panel[panel['in_baseline']].copy()
base['log_failures'] = np.log(base['failures'])
base['fake_post'] = (base['date'] >= '2018-01-01').astype(int)
base['fake_did'] = base['fake_post'] * base['treated']
m3 = smf.ols('log_failures ~ C(sector) + fake_post + fake_did', data=base).fit(
    cov_type='cluster', cov_kwds={'groups': base['sector']})
print(m3.summary().tables[1])
print(f"\nPlacebo DiD: {m3.params['fake_did']:+.3f} log pts "
      f"(p = {m3.pvalues['fake_did']:.3f})")

## 4. DiD decomposition

In [ ]:
g = sample.groupby(['treated', 'post'])['log_failures'].mean().unstack()
before_t, after_t = g.loc[1, 0], g.loc[1, 1]
before_c, after_c = g.loc[0, 0], g.loc[0, 1]
counterfactual = before_t + (after_c - before_c)

fig, ax = plt.subplots(figsize=(7, 5))
x = [0, 1]
ax.plot(x, [before_t, after_t], 'o-', color='#c0392b', lw=2, label='Treated (bank-dependent)')
ax.plot(x, [before_c, after_c], 'o-', color='#7f8c8d', lw=2, label='Control')
ax.plot(x, [before_t, counterfactual], 'o--', color='#c0392b', lw=1.5, alpha=0.5,
        label='Counterfactual')
ax.annotate('', xy=(1.04, after_t), xytext=(1.04, counterfactual),
            arrowprops=dict(arrowstyle='<->', color='black', lw=1.4))
ax.text(1.06, (after_t + counterfactual) / 2,
        f'DiD = {(after_t - counterfactual):+.3f} log', va='center', fontsize=10)
ax.set_xticks(x)
ax.set_xticklabels(['Baseline\n(2015-2019)', 'Post-hike\n(2022-07 on)'])
ax.set_ylabel('Mean log failures')
ax.set_title('DiD decomposition - bank-dependent vs control sectors')
ax.legend(loc='upper left')
ax.set_xlim(-0.2, 1.4)
plt.tight_layout()
plt.savefig('../outputs/did_decomposition.png', dpi=150, bbox_inches='tight')
plt.show()

## Results summary

- The main DiD estimates the extra change in failures for bank-credit-dependent sectors after July 2022, net of the common post-pandemic movement.
- The tertile split (sharper treatment/control contrast) and the baseline placebo are reported alongside as robustness.

**Limitations specific to this analysis:**
1. **Nine sectors only** — cluster-robust inference rests on few clusters; treat p-values as indicative.
2. **12-month cumulative outcome** — the July 2022 break enters gradually over the following year, so the estimate reflects a smoothed annual effect rather than an instantaneous jump.
3. **Treatment proxy** — bank-debt share stands in for short-term-rate exposure, which is not measured directly by sector in open data.
4. The territorial (ZFRR) question is addressed separately, at department level.